In [ ]:
# Cell 1: Hardware / CUDA / BF16 / FP8 Check + Google Drive Persistent Checkpoint Mount
import os
from pathlib import Path

import torch

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024**3)
    supports_bf16 = props.major >= 8
    supports_fp8 = (props.major == 8 and props.minor == 9) or (props.major >= 9)
    print(f"GPU: {props.name} | VRAM: {vram_gb:.2f} GiB | CC: {props.major}.{props.minor}")
    print(f"BF16 Tensor Cores: {supports_bf16} | Native FP8 (Ada L4 / Hopper): {supports_fp8}")
else:
    print("WARNING: No CUDA GPU detected. Switch Colab Runtime -> Change runtime type -> L4 or A100.")

USE_GOOGLE_DRIVE = True
CKPT_BASE = Path("/content/checkpoints")
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        CKPT_BASE = Path("/content/drive/MyDrive/LucidMachines/checkpoints")
    except Exception as exc:
        print(f"Drive mount skipped ({exc}); using local {CKPT_BASE}")

CKPT_BASE.mkdir(parents=True, exist_ok=True)
os.environ["LUCID_MACHINES_CKPT_DIR"] = str(CKPT_BASE)
print(f"Persistent Checkpoint Root: {CKPT_BASE}")

In [ ]:
# Cell 2: Install Lucid Machines v1.0.0 (Branch: release/lucid-machines-v1.0.0) & Verify Configs
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/TriuneTransformer")
REPO_URL = os.environ.get("LUCID_MACHINES_REPO_URL", "https://github.com/lucid-machines/triune.git")
BRANCH = "release/lucid-machines-v1.0.0"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

%cd /content/TriuneTransformer
!pip install -q --upgrade pip
!pip install -q -e ".[dev]" || pip install -q -e .
!python3 scripts/train.py --model_config junkie-3b --dry-run --batch-size 4 --seq-len 1024 --rank 64 --steer-dirs 4 --use_fp8
!python3 scripts/train.py --model_config junkie-1b --dry-run --batch-size 4 --seq-len 1024 --rank 64 --steer-dirs 4

In [ ]:
# Cell 3: Train Junkie-3B (2.99B Total / 16 Experts) on Colab L4 (24GB), G4 (48GB+), or A100 (80GB)
# Hardware Memory Reality (CentroidSteer + Muon + Base + BF16 grads):
#   - Total Params: 2,990,355,155 (2.99B) | 16 Routed Experts + 1 Shared Expert (top_k_experts=1)
#   - Active Params: 395.74M MoE-trunk Cortex (469.69M w/ embed) | 206.73M avg dynamic (Reflex -> Limbic -> Cortex)
#   - On L4 (24GB), auto-enables `--streaming --use_fp8` and reduced subspace `--rank 16` with expandable_segments:True.
#   - On A100-80GB (BF16) or G4 (>40GB VRAM), runs full resident with `--rank 64` (`--use_fp8` only when Ada/Blackwell/Hopper).
#   - Refuses T4 (<20GB) up front so compute units are never wasted on an OOM.
import os
from pathlib import Path
import torch

%cd /content/TriuneTransformer
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ckpt_3b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-3b"
ckpt_3b.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Switch Colab Runtime -> L4, G4, or A100.")
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
if vram_gb < 20.0:
    raise RuntimeError(f"GPU {props.name} ({vram_gb:.1f} GiB) is too small for Junkie-3B; switch Colab Runtime to L4 (24GB), G4, or A100.")
supports_fp8 = (props.major == 8 and props.minor == 9) or (props.major >= 9)
is_l4_tier = vram_gb < 40.0
rank_val = 16 if is_l4_tier else 64
batch_val = 1 if is_l4_tier else 4
seq_val = 512 if is_l4_tier else 1024
fp8_flag = "--use_fp8" if supports_fp8 else ""
stream_flag = "--streaming --use_8bit_base" if is_l4_tier else ""

CURRICULUM_STAGE = os.environ.get("JUNKIE_CURRICULUM_STAGE", "stage1_foundation")
print(f"Launching Junkie-3B | {props.name} ({vram_gb:.1f} GiB) | fp8={supports_fp8} | rank={rank_val} | streaming={is_l4_tier} | Stage={CURRICULUM_STAGE}")

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/train.py \
    --model_config models/configs/junkie-3b.json \
    --checkpoint_dir "{ckpt_3b}" \
    --auto_resume \
    --depth_mode dynamic \
    --target_depth_dist 0.50,0.30,0.20 \
    --steps 10000 \
    --batch_size {batch_val} \
    --grad_accum_steps 8 \
    --seq_len {seq_val} \
    --rank {rank_val} \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login \
    {fp8_flag} \
    {stream_flag}

In [ ]:
# Cell 3B: Loop-Expand Trained Junkie-3B (16L / 2.99B) -> Junkie-7B-DUS (36L / ~7.02B) & Heal
# Preserves dense prefix + Reflex L2 bit-exact, loops best middle MoE core into 36L, heals with WSD.
import os
from pathlib import Path
import torch

%cd /content/TriuneTransformer
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ckpt_root = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints"))
src_3b = ckpt_root / "junkie-3b" / "latest.pt"
if not src_3b.is_file():
    raise FileNotFoundError(f"Train Junkie-3B first (missing {src_3b}).")
ckpt_7b_dus = ckpt_root / "junkie-7b-dus"
ckpt_7b_dus.mkdir(parents=True, exist_ok=True)
init_ckpt = ckpt_7b_dus / "expanded_init.pt"

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
if vram_gb < 20.0:
    raise RuntimeError(f"GPU {props.name} ({vram_gb:.1f} GiB) too small; switch Runtime to A100, G4, or L4 High-RAM.")
supports_fp8 = (props.major == 8 and props.minor == 9) or (props.major >= 9)
fp8_arg = "--use-fp8" if supports_fp8 else "--no-use-fp8"
fp8_train_flag = "--use_fp8" if supports_fp8 else ""
stream_flag = "--streaming --use_8bit_base" if vram_gb < 75.0 else ""
rank_val = 256 if vram_gb >= 75.0 else (64 if vram_gb >= 40.0 else 32)
seq_val = 1024 if vram_gb >= 75.0 else 512

if not (ckpt_7b_dus / "latest.pt").is_file() and not init_ckpt.is_file():
    !python3 scripts/expand_depth.py \
        --input "{src_3b}" \
        --output "{init_ckpt}" \
        --target-config models/configs/junkie-7b-dus.json \
        --strategy best_loop \
        --smooth-scale 0.25 \
        {fp8_arg}

resume_args = "--auto_resume" if (ckpt_7b_dus / "latest.pt").is_file() else f"--resume_latest \"{init_ckpt}\" --resume_best"
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/train.py \
    --model_config models/configs/junkie-7b-dus.json \
    --checkpoint_dir "{ckpt_7b_dus}" \
    {resume_args} \
    --depth_mode joint \
    --steps 3000 \
    --batch_size 1 \
    --grad_accum_steps 16 \
    --seq_len {seq_val} \
    --rank {rank_val} \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 100 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login \
    {fp8_train_flag} \
    {stream_flag}

In [ ]:
# Cell 4: Train Junkie-1B (1.16B Total / 374.99M Expected Active / 317.48M Dynamic Active)
# Exact measured static footprint: 5.87 GiB BF16 (4.79 GiB FP8, 0.1992 B/p on experts) vs 12.99 GiB FP32 AdamW.
import os
from pathlib import Path

%cd /content/TriuneTransformer
ckpt_1b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-1b"
ckpt_1b.mkdir(parents=True, exist_ok=True)

!python3 scripts/train.py \
    --model_config models/configs/junkie-1b.json \
    --checkpoint_dir "{ckpt_1b}" \
    --auto_resume \
    --use_fp8 \
    --depth_mode dynamic \
    --steps 8000 \
    --batch_size 4 \
    --grad_accum_steps 8 \
    --seq_len 1024 \
    --rank 64 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login

In [ ]:
# Cell 5: Train Junkie-Mini (535.10M Total / 230.81M Expected Active) for Rapid Single-GPU Iteration
# Fits in ~3.12 GiB static VRAM (CentroidSteer rank=64: 0.1992 B/p on experts) — runs on L4, T4, or A100.
import os
from pathlib import Path

%cd /content/TriuneTransformer
ckpt_mini = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-mini"
ckpt_mini.mkdir(parents=True, exist_ok=True)

!python3 scripts/train.py \
    --model_config models/configs/junkie-mini.json \
    --checkpoint_dir "{ckpt_mini}" \
    --auto_resume \
    --steps 5000 \
    --batch_size 4 \
    --grad_accum_steps 8 \
    --seq_len 1024 \
    --rank 64 \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 200 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login

In [ ]:
# Cell 6: Train Flagship Junkie-7B (7.40B Total / 1.95B Expected Active) on Single A100-80GB / G4 / L4 High-RAM
import os
from pathlib import Path
import torch

%cd /content/TriuneTransformer
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
ckpt_7b = Path(os.environ.get("LUCID_MACHINES_CKPT_DIR", "/content/checkpoints")) / "junkie-7b"
ckpt_7b.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. Switch Colab Runtime -> A100, G4, or L4 (High-RAM).")
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024**3)
mem_lines = Path("/proc/meminfo").read_text().splitlines() if Path("/proc/meminfo").is_file() else []
host_ram_gb = next((int(l.split()[1]) / (1024**2) for l in mem_lines if l.startswith("MemTotal:")), 32.0)
if vram_gb < 20.0:
    raise RuntimeError(f"GPU {props.name} ({vram_gb:.1f} GiB) cannot run Junkie-7B; switch Runtime to A100, G4, or L4 (High-RAM).")
supports_fp8 = (props.major == 8 and props.minor == 9) or (props.major >= 9)
if vram_gb >= 75.0:
    stream_flag, rank_val, seq_val = "", 256, 1024
elif vram_gb >= 40.0:
    stream_flag, rank_val, seq_val = "--streaming --use_8bit_base", 64, 512
else:
    if host_ram_gb < 22.0:
        raise RuntimeError(f"L4 ({vram_gb:.1f} GiB) with {host_ram_gb:.1f} GiB host RAM needs Colab High-RAM enabled for Junkie-7B streaming.")
    stream_flag, rank_val, seq_val = "--streaming --use_8bit_base", 32, 512
fp8_flag = "--use_fp8" if supports_fp8 else ""
print(f"Junkie-7B | {props.name} ({vram_gb:.1f} GiB) | RAM={host_ram_gb:.1f} GiB | fp8={supports_fp8} | rank={rank_val} | flags={stream_flag or 'resident'}")

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python3 scripts/train.py \
    --model_config models/configs/junkie-7b.json \
    --checkpoint_dir "{ckpt_7b}" \
    --auto_resume \
    --steps 2000 \
    --batch_size 1 \
    --grad_accum_steps 16 \
    --seq_len {seq_val} \
    --rank {rank_val} \
    --steer_dirs 4 \
    --steer_mode centroid \
    --save_every 100 \
    --eval_every 200 \
    --log_every 10 \
    --no_wandb \
    --no_hf_login \
    {fp8_flag} \
    {stream_flag}

In [ ]:
# Cell 7: CentroidSteer 5-Seed FineWeb-Edu Scaling Validation vs Scientific Baselines
# Compares CentroidSteer against No-Steer Low-Rank, Memory-Matched Random Subspace, and Full-Rank AdamW.
%cd /content/TriuneTransformer
!python3 benchmarks/validate_centroid_steer_upgrade.py \
    --seeds 42 1337 2026 777 1024 \
    --md-steps 120 \
    --fw-steps 250 \
    --output-json benchmarks/colab_centroidsteer_results.json \
    --output-report benchmarks/COLAB_CENTROIDSTEER_REPORT.md

In [ ]:
# Cell 8: Launch Polysom Studio v1.0.0 Live on Cloud GPU (Colab Port Forward / Window / Iframe)
import subprocess
import time
import urllib.request

%cd /content/TriuneTransformer
studio_proc = subprocess.Popen(
    ["kcx", "serve", "--host", "127.0.0.1", "--port", "8000"],
    stdout=open("/tmp/polysom_studio.log", "w"),
    stderr=subprocess.STDOUT,
)

for _ in range(30):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/v1/telemetry/gpu", timeout=1) as r:
            if r.status == 200:
                break
    except Exception:
        time.sleep(1)

try:
    from google.colab import output as colab_output

    proxy_url = colab_output.eval_js("google.colab.kernel.proxyPort(8000)")
    print(f"Polysom Studio v1.0.0 live proxy URL: {proxy_url}")
    colab_output.serve_kernel_port_as_window(8000)
    colab_output.serve_kernel_port_as_iframe(8000, height="760")
except Exception as exc:
    print(f"Polysom Studio v1.0.0 running at http://127.0.0.1:8000 (Colab proxy unavailable: {exc})")